# Silver Layer - Confluence Data Transformation

Reads Bronze Parquet files, cleanses data:
- Strips HTML from page/comment bodies
- Parses and normalizes dates
- Computes word count and content length
- Deduplicates by primary key

Writes both Parquet and CSV (CSV required for AI Search indexing).

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'beautifulsoup4>=4.12.0'])
print('Dependencies installed.')

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from bs4 import BeautifulSoup

BASE = '/lakehouse/default/Files'

def html_to_text(html):
    if not html or not isinstance(html, str):
        return ''
    return BeautifulSoup(html, 'html.parser').get_text(separator=' ', strip=True)

def read_bronze(table):
    path = f'{BASE}/bronze/{table}/data.parquet'
    if os.path.exists(path):
        return pd.read_parquet(path)
    return pd.DataFrame()

def write_silver(df, table):
    out_dir = f'{BASE}/silver/{table}'
    os.makedirs(out_dir, exist_ok=True)
    df.to_parquet(f'{out_dir}/data.parquet', index=False)
    df.to_csv(f'{out_dir}/data.csv', index=False)
    print(f'  Wrote {len(df)} rows to silver/{table} (Parquet + CSV)')

In [ ]:
# ── Transform Spaces ─────────────────────────────────────────────
spaces = read_bronze('confluence_spaces')
if not spaces.empty:
    meta = [c for c in spaces.columns if c.startswith('_')]
    spaces = spaces.drop(columns=meta, errors='ignore')
    spaces['space_name'] = spaces['space_name'].str.strip()
    spaces['space_key'] = spaces['space_key'].str.upper().str.strip()
    spaces['space_type'] = spaces['space_type'].str.lower().str.strip()
    spaces['description'] = spaces['description'].apply(html_to_text)
    spaces['_processed_at'] = datetime.now(timezone.utc).isoformat()
    write_silver(spaces, 'confluence_spaces')
else:
    print('No spaces data found in bronze.')

In [ ]:
# ── Transform Pages ──────────────────────────────────────────────
pages = read_bronze('confluence_pages')
if not pages.empty:
    meta = [c for c in pages.columns if c.startswith('_')]
    pages = pages.drop(columns=meta, errors='ignore')
    pages['body_text'] = pages['body_html'].apply(html_to_text)
    pages['word_count'] = pages['body_text'].apply(
        lambda x: len(x.split()) if isinstance(x, str) else 0
    )
    pages['content_length'] = pages['body_text'].str.len().fillna(0).astype(int)
    pages['created_date'] = pd.to_datetime(pages['created_date'], errors='coerce')
    pages['last_updated_date'] = pd.to_datetime(pages['last_updated_date'], errors='coerce')
    pages['title'] = pages['title'].str.strip()
    pages['space_key'] = pages['space_key'].str.upper().str.strip()
    pages['status'] = pages['status'].str.lower().str.strip()
    before = len(pages)
    pages = pages.drop_duplicates(subset=['page_id'], keep='last')
    print(f'  Deduped pages: removed {before - len(pages)} duplicates')
    pages['_processed_at'] = datetime.now(timezone.utc).isoformat()
    write_silver(pages, 'confluence_pages')
else:
    print('No pages data found in bronze.')

In [ ]:
# ── Transform Comments ───────────────────────────────────────────
comments = read_bronze('confluence_comments')
if not comments.empty:
    meta = [c for c in comments.columns if c.startswith('_')]
    comments = comments.drop(columns=meta, errors='ignore')
    comments['comment_text'] = comments['body_html'].apply(html_to_text)
    comments['word_count'] = comments['comment_text'].apply(
        lambda x: len(x.split()) if isinstance(x, str) else 0
    )
    comments['created_date'] = pd.to_datetime(comments['created_date'], errors='coerce')
    comments['author'] = comments['author'].str.strip()
    before = len(comments)
    comments = comments.drop_duplicates(subset=['comment_id'], keep='last')
    print(f'  Deduped comments: removed {before - len(comments)} duplicates')
    comments['_processed_at'] = datetime.now(timezone.utc).isoformat()
    write_silver(comments, 'confluence_comments')
else:
    print('No comments data found in bronze.')

print('Silver layer complete.')